# 08 — Prepare shared features and raw-count scVI input

Only rows in one explicit integration_group are combined. Different studies are not automatically integrated. Preparation is sample-at-a-time; training is not fully streamed. The two-pass HVG method is preserved from v1.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "vhd" / "control").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter in the extracted pipeline folder (or its notebooks folder).")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from vhd.control.manifest import load_project, check_policy, save_plan, sample_layout
MANIFEST = Path(os.environ.get("VHD_MANIFEST", ROOT / "config" / "samples.csv"))
SETTINGS = Path(os.environ.get("VHD_SETTINGS", ROOT / "config" / "settings.json"))
if not MANIFEST.exists() or not SETTINGS.exists():
    raise FileNotFoundError("Copy a supplied samples.*.csv to config/samples.csv and settings.g5_24xlarge.example.json to config/settings.json; edit paths and policy first.")
PROJECT = load_project(MANIFEST, SETTINGS)
check_policy(PROJECT)
# Default is a dry run. Set True here only after reviewing the printed plan.
EXECUTE = os.environ.get("VHD_EXECUTE", "0") == "1"
# Optional pilot selection, e.g. ["StudyLegacy__Sample01"]. None selects all applicable rows.
SAMPLE_KEYS = None


## Select one integration group; each group gets its own model and output destinations

In [ ]:
groups = sorted({r["integration_group"] for r in PROJECT["rows"] if r["integration_group"]})
print("Groups:", groups)
if not groups: raise ValueError("Fill integration_group for at least two compatible cell-producing samples.")
GROUP = groups[0]  # edit explicitly for another group
from vhd.compute.launch import launch_integration
launch_integration(PROJECT, GROUP, 'prepare', execute=EXECUTE)